# EDA Bank Marketing


In [ ]:
import pandas as pd
import os
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# Load Dataset

In [ ]:
path = os.path.join(os.getcwd(), "..", "..", "datasets", "bank-additional-full.csv")
data = pd.read_csv(path, sep=";")
data.shape

# Initial Exploration

In [ ]:
data.info()

In [ ]:
data.describe()

In [ ]:
data.columns

In [ ]:
data["y"].value_counts().plot(kind="bar")

-> highly imbalanced

## Mising values

In [ ]:
data.isna().sum()

### Duplicates

In [ ]:
data.duplicated().sum()
print("Number of duplicate rows:", data.duplicated().sum())

# Univariate Analysis

## Categorical Variables

In [ ]:
cat_features = data.select_dtypes(include=["object"]).columns.to_list()

In [ ]:
for col in cat_features:
    plt.figure(figsize=(10, 5))
    sns.countplot(x=col, data=data)
    plt.title(f"Distribution of {col}")
    plt.xticks(rotation=45)
    plt.show()
    print(f"Value counts for {col}:")
    print(data[col].value_counts())
    print("\n")

In [ ]:
data.replace("unknown", np.nan, inplace=True)
data.replace("nonexistent", np.nan, inplace=True)

In [ ]:
# calculate the percetage of nan values in each categorical feature
nan_percentage = data[cat_features].isna().mean() * 100
print("Percentage of NaN values in each categorical feature:")
print(nan_percentage)

## Numerical Features

In [ ]:
num_features = data.select_dtypes(include=["int64", "float64"]).columns.to_list()

In [ ]:
for col in num_features:
    plt.figure(figsize=(10, 5))
    sns.histplot(data[col], kde=True)
    plt.title(f"Distribution of {col}")
    plt.show()

In [ ]:
nan_percentage = data[num_features].isna().mean() * 100
print("Percentage of NaN values in each categorical feature:")
print(nan_percentage)

# Bivariate Analysis

## Categprical Features x target

In [ ]:
r = len(cat_features)

In [ ]:
# Defining a function to Notate the percent count of each value on the bars
def annot_percent(axes):
    """Takes axes as input and labels the percent count of each bar in a countplot"""
    for p in plot.patches:
        total = sum(p.get_height() for p in plot.patches) / 100
        percent = round((p.get_height() / total), 2)
        x = p.get_x() + p.get_width() / 2
        y = p.get_height()
        plot.annotate(f"{percent}%", (x, y), ha="center", va="bottom")

In [ ]:
plt.figure(figsize=(18, r * 4))
for n, var in enumerate(cat_features):
    plot = plt.subplot(r, 1, n + 1)
    sns.countplot(x=data[var], hue=data["y"]).margins(y=0.15)
    plt.title(f"{var.title()}", weight="bold")
    plt.tight_layout()
    annot_percent(plot)

## Numerical x terget

In [ ]:
# bivariate analysis: numerical features vs target variable
def bivariate_cat(X, y, feature):
    data = pd.concat([X[[feature]], y], axis=1)
    data.columns = [feature, "y"]

    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    sns.barplot(data=data, x=feature, y="y", ax=axes[0])
    axes[0].set_title(f"Mean cnt by {feature}")
    sns.boxplot(data=data, x=feature, y="y", ax=axes[1])
    axes[1].set_title(f"Boxplot cnt by {feature}")
    # change violinplot to displot
    sns.histplot(data=data, x=feature, hue="y", kde=True, ax=axes[2])
    axes[2].set_title(f"Distribution cnt by {feature}")
    plt.tight_layout()
    plt.show()


X = data.drop(columns=["y"])
for feat in num_features:
    print(f"── {feat} ──")
    bivariate_cat(X, data["y"], feat)

## Multivariate Analysis

In [ ]:
# heatmap of correlation between numerical features
plt.figure(figsize=(12, 8))
sns.heatmap(data[num_features].corr(), annot=True, cmap="coolwarm", fmt=".2f")

In [ ]:
# pairplot of numerical features
sns.pairplot(pd.concat([X[num_features], data["y"]], axis=1))
plt.show()

### **Key Observations:**
- the dataset has 41188 raws and 21 columns
- the dataset is highly imbalanced
- there are 12 duplicates.

**Categorical Features:**

- `poutcome`, `default`, `education`, `housing `, `loan`,`job` and `marital`all have null values of 86.3%, 20.8%, 4.2%, 2.4%, 2.4%, 0.8% and 0.19%, respectively.

**Numerical Features:**

- there is no missing values in numerical features.
- `age`,`duration` and `campaign` contain outliers.
- `previous` and `pdays` have a **flat interquartile rang**.
- `cons.conf.idx` has one outlier.
- other numeric variables has no outlier.

**Multivariate Analysis:**

- `euribor3m` and `emp.var.rate` are highly correlated.
- `nr.employed` and `euribor3m` also highly correlated


### **Changes to be made:**

- remove duplicates.

**Categorical Features:**

- replace null values with their equivalent modes for features that has few null values,like `job`.
- remove features with more than 50% null values because they were useless negatively impacted model performance.

**Numerical Features :**

- remove outliers from `age`,`duration`, `campaign`and `cons.conf.idx`.
- remove flat IQR features :(`previous`,`pdays`)